# Самостоятельная работа: языковая модель на Elman RNN

Корпус: `maxzt/RuHeritage-Corpus`, каталог `data`, поле `text`, закреплённая ревизия. `nn.RNN` не используется.

## Этап 1. Настройка

In [ ]:
from __future__ import annotations

import copy
import math
import random
import re
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

SEED = 42
DATASET_NAME = 'maxzt/RuHeritage-Corpus'
DATASET_REVISION = 'b2d01eaad3ddb119fe10c4cb17a86e47415a77cb'
DATASET_DATA_DIR = 'data'
TEXT_FIELD = 'text'
START_DOCUMENT = 0
MAX_DOCUMENTS = 1_200
MAX_WORDS_PER_DOCUMENT = 160
MAX_ORDINARY_WORDS = 6_000
MAX_SEQUENCE_LENGTH = 32
EMBEDDING_DIM = 128
HIDDEN_DIM = 128
NUM_LAYERS = 2
BATCH_SIZE = 64
LEARNING_RATE = 2e-3
WEIGHT_DECAY = 1e-4
MAX_EPOCHS = 12
PATIENCE = 3

SPECIAL_TOKENS = ['<pad>', '<unk>', '<bos>', '<eos>']
PAD_IDX, UNK_IDX, BOS_IDX, EOS_IDX = range(4)
DEVICE = torch.device('cpu')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(max(1, min(4, torch.get_num_threads())))
print(f'PyTorch: {torch.__version__}; устройство: {DEVICE}')

## Этап 2. Загрузка и нормализация

In [ ]:
WORD_RE = re.compile(r'[а-яё]+(?:-[а-яё]+)?', flags=re.IGNORECASE)


def normalize_and_tokenize(text, max_words):
    normalized = ' '.join(str(text).lower().split())
    return WORD_RE.findall(normalized)[:max_words]


def load_documents(text_field, start_document, max_documents, max_words):
    stream = load_dataset(
        DATASET_NAME,
        revision=DATASET_REVISION,
        data_dir=DATASET_DATA_DIR,
        split='train',
        streaming=True,
        trust_remote_code=True,
    )
    train, val = [], []
    stop = start_document + max_documents
    for i, row in enumerate(stream):
        if i < start_document:
            continue
        if i >= stop:
            break
        tokens = normalize_and_tokenize(row.get(text_field, ''), max_words)
        if len(tokens) < 2:
            continue
        (val if (i - start_document) % 6 == 0 else train).append(tokens)
    if not train or not val:
        raise RuntimeError('Нет обеих частей подвыборки.')
    return train, val


train_documents, val_documents = load_documents(
    TEXT_FIELD, START_DOCUMENT, MAX_DOCUMENTS, MAX_WORDS_PER_DOCUMENT,
)
print(f'Обучающих документов: {len(train_documents)}')
print(f'Валидационных документов: {len(val_documents)}')
print('Пример:', train_documents[0][:20])

## Этап 3. Словарь и сдвинутые пары

In [ ]:
def build_vocabulary(documents, max_ordinary_words):
    frequencies = Counter(t for doc in documents for t in doc)
    ordinary = [t for t, _ in frequencies.most_common(max_ordinary_words)]
    itos = SPECIAL_TOKENS + ordinary
    stoi = {t: i for i, t in enumerate(itos)}
    return stoi, itos, frequencies


stoi, itos, frequencies = build_vocabulary(train_documents, MAX_ORDINARY_WORDS)
VOCAB_SIZE = len(itos)
assert [stoi[t] for t in SPECIAL_TOKENS] == [0, 1, 2, 3]
assert stoi.get('несуществующее_слово', UNK_IDX) == UNK_IDX
print('Размер словаря:', VOCAB_SIZE)


def encode_document(tokens):
    return [BOS_IDX] + [stoi.get(t, UNK_IDX) for t in tokens] + [EOS_IDX]


def make_language_pairs(documents):
    pairs = []
    for doc in documents:
        encoded = encode_document(doc)
        for start in range(0, len(encoded) - 1, MAX_SEQUENCE_LENGTH):
            window = encoded[start:start + MAX_SEQUENCE_LENGTH + 1]
            if len(window) >= 2:
                pairs.append((
                    torch.tensor(window[:-1], dtype=torch.long),
                    torch.tensor(window[1:], dtype=torch.long),
                ))
    return pairs


train_pairs = make_language_pairs(train_documents)
val_pairs = make_language_pairs(val_documents)
print('Обучающих фрагментов:', len(train_pairs))
print('Валидационных фрагментов:', len(val_pairs))

## Этап 4. Мини-пакет и PAD

In [ ]:
class LanguageModelDataset(Dataset):
    def __init__(self, pairs):
        self.pairs = pairs
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        return self.pairs[i]


def collate_language_batch(batch):
    inputs, targets = zip(*batch)
    return (
        pad_sequence(inputs, batch_first=True, padding_value=PAD_IDX),
        pad_sequence(targets, batch_first=True, padding_value=PAD_IDX),
    )


train_loader = DataLoader(
    LanguageModelDataset(train_pairs), batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=collate_language_batch, num_workers=0,
    generator=torch.Generator().manual_seed(SEED),
)
val_loader = DataLoader(
    LanguageModelDataset(val_pairs), batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=collate_language_batch, num_workers=0,
)
sample_input, sample_target = next(iter(train_loader))
assert sample_input.shape == sample_target.shape
assert sample_input.shape[1] <= MAX_SEQUENCE_LENGTH
assert int((sample_target != PAD_IDX).sum()) > 0
print('Форма мини-пакета:', tuple(sample_input.shape))

## Этап 5. Ручная Elman RNN

In [ ]:
class ElmanRNNUnit(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.W_x = nn.Parameter(torch.empty(hidden_size, input_size))
        self.U_h = nn.Parameter(torch.empty(hidden_size, hidden_size))
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        nn.init.xavier_uniform_(self.W_x)
        nn.init.orthogonal_(self.U_h)
    def forward(self, x_t, h_prev):
        return torch.tanh(
            F.linear(x_t, self.W_x, self.b_h) + F.linear(h_prev, self.U_h)
        )


class ElmanRNN(nn.Module):
    def __init__(self, hidden_size, num_layers):
        super().__init__()
        self.hidden_size = hidden_size
        self.layers = nn.ModuleList(
            [ElmanRNNUnit(hidden_size, hidden_size) for _ in range(num_layers)]
        )
    def forward(self, embeddings):
        B, T, _ = embeddings.shape
        states = [embeddings.new_zeros(B, self.hidden_size) for _ in self.layers]
        outputs = []
        for t in range(T):
            x = embeddings[:, t, :]
            new_states = []
            for layer, prev in zip(self.layers, states):
                x = layer(x, prev)
                new_states.append(x)
            states = new_states
            outputs.append(x)
        return torch.stack(outputs, dim=1)


class ElmanLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, EMBEDDING_DIM, padding_idx=PAD_IDX)
        self.rnn = ElmanRNN(HIDDEN_DIM, NUM_LAYERS)
        self.output = nn.Linear(HIDDEN_DIM, vocab_size)
    def forward(self, input_ids):
        return self.output(self.rnn(self.embedding(input_ids)))


model = ElmanLanguageModel(VOCAB_SIZE).to(DEVICE)
with torch.no_grad():
    emb = model.embedding(sample_input.to(DEVICE))
    hid = model.rnn(emb)
    logits = model(sample_input.to(DEVICE))
assert emb.shape == (sample_input.shape[0], sample_input.shape[1], EMBEDDING_DIM)
assert hid.shape == emb.shape
assert logits.shape == (sample_input.shape[0], sample_input.shape[1], VOCAB_SIZE)
print('эмбеддинги', tuple(emb.shape))
print('скрытые', tuple(hid.shape))
print('логиты', tuple(logits.shape))

## Этап 6. Обучение и ранняя остановка

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, reduction='sum')
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                               weight_decay=WEIGHT_DECAY)


def negative_log_likelihood(logits, targets):
    V = logits.shape[-1]
    summed = criterion(logits.reshape(-1, V), targets.reshape(-1))
    count = int((targets != PAD_IDX).sum())
    if count == 0:
        raise RuntimeError('Нет содержательных токенов.')
    return summed, count


def run_epoch(loader, training):
    model.train(training)
    total, tokens = 0.0, 0
    iterator = tqdm(loader, leave=False, disable=not training)
    for x, y in iterator:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if training:
            optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            logits = model(x)
            nll, count = negative_log_likelihood(logits, y)
            loss = nll / count
            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
        total += float(nll.detach().cpu())
        tokens += count
    epoch_loss = total / tokens
    if not math.isfinite(epoch_loss):
        raise FloatingPointError('Потеря не конечна.')
    return epoch_loss


history = {'loss': [], 'val_loss': []}
best_val, best_state, patience_counter = float('inf'), None, 0
artifact = Path('artifacts/best_elman_rnn_ruheritage.pt')
artifact.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(1, MAX_EPOCHS + 1):
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    history['loss'].append(tr); history['val_loss'].append(va)
    print(f'Эпоха {epoch:02d}: loss={tr:.4f}; val_loss={va:.4f}')
    if va < best_val:
        best_val = va
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, artifact)
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Ранняя остановка.')
            break
model.load_state_dict(best_state)
print('Лучшее состояние:', artifact)

## Этап 7. Оценка, перплексия, top-k, генерация

In [ ]:
final_val = run_epoch(val_loader, False)
ppl = math.exp(min(final_val, 20.0))
assert math.isfinite(ppl) and ppl > 0

plt.figure(figsize=(7, 4))
eps = range(1, len(history['loss']) + 1)
plt.plot(eps, history['loss'], marker='o', label='loss')
plt.plot(eps, history['val_loss'], marker='o', label='val_loss')
plt.xlabel('Эпоха'); plt.ylabel('Кросс-энтропия на токен')
plt.grid(alpha=0.3); plt.legend(); plt.show()
print(f'Валидационная потеря: {final_val:.4f}')
print(f'Валидационная перплексия: {ppl:.2f}')


def prompt_to_ids(prompt):
    tokens = normalize_and_tokenize(prompt, MAX_SEQUENCE_LENGTH - 1)
    return [BOS_IDX] + [stoi.get(t, UNK_IDX) for t in tokens]


@torch.no_grad()
def top_k_next_words(prompt, k=5):
    ids = torch.tensor([prompt_to_ids(prompt)], dtype=torch.long)
    probs = torch.softmax(model(ids)[0, -1], dim=-1)
    values, indices = torch.topk(probs, k=k)
    return [(itos[i.item()], v.item()) for v, i in zip(values, indices)]


@torch.no_grad()
def greedy_generate(prompt, max_new_tokens=12):
    ids = prompt_to_ids(prompt)
    for _ in range(max_new_tokens):
        inp = torch.tensor([ids], dtype=torch.long)
        nxt = int(torch.argmax(model(inp)[0, -1]))
        if nxt == EOS_IDX:
            break
        ids.append(nxt)
    return ' '.join(itos[i] for i in ids if i not in {PAD_IDX, BOS_IDX, EOS_IDX})


prompt = ' '.join(train_documents[0][:3])
print('Запрос:', repr(prompt))
for token, p in top_k_next_words(prompt, k=5):
    print(f'  {token:>18s}: {p:.4f}')
print('Жадная генерация:', greedy_generate(prompt))

## Этап 8. Самопроверка

- RuHeritage-Corpus, поле `text`, каталог `data`, закреплённая ревизия.
- Словарь только по обучающей части; индексы 0–3 — PAD/UNK/BOS/EOS.
- `input_ids/target_ids` формы `(B,T)`, `T ≤ 32`.
- Эмбеддинги и скрытые — `(B,T,128)`, логиты — `(B,T,V)`.
- PAD исключён из кросс-энтропии.
- Потери и перплексия конечны; сохранено лучшее состояние.
- Нет TensorFlow/Keras, нет `nn.RNN`.